In [2]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("../dataset/league_data.db")

# List all tables
tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", conn)
print("=== TABLES ===")
print(tables)

# For each table, show columns and a sample row
for table in tables["name"]:
    df = pd.read_sql(f"SELECT * FROM {table} LIMIT 2", conn)
    print(f"\n=== {table} ===")
    print(f"Columns: {list(df.columns)}")
    print(df.to_string())

conn.close()

=== TABLES ===
            name
0  dim_champions
1      dim_items
2      dim_runes
3    dim_players
4        matches
5   player_stats
6     team_stats
7           bans
8   player_items
9   player_runes

=== dim_champions ===
Columns: ['id', 'name']
    id    name
0  246  Qiyana
1   24     Jax

=== dim_items ===
Columns: ['id', 'name']
     id              name
0  2504    Kaenic Rookern
1  3047  Plated Steelcaps

=== dim_runes ===
Columns: ['id', 'name']
     id                  name
0  8437  Grasp of the Undying
1  8401           Shield Bash

=== dim_players ===
Columns: ['puuid', 'name', 'tag']
                                                                            puuid     name   tag
0  rJeiFt6iWmavjuIeMhCTUPD6gG2QyWNxzbNuiw3eagMXEtUYtU5EkU6h-MOHATYJCcXfbwWEIhpqoQ  DRX 왕눈이   KR1
1  1RrlRlccxS-93MIOt6_MhQD5Z8YOIyEcOLtAglk_dVgjSCLDnawAtUPLdFJgc_u3V9Ocn-NQKoaemA      멍다1  yaha

=== matches ===
Columns: ['match_id', 'patch', 'duration', 'timestamp', 'match_type']
        match_id  p

In [3]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("../dataset/league_data.db")

for table in ["matches", "player_stats", "player_items", "player_runes", "bans"]:
    count = pd.read_sql(f"SELECT COUNT(*) as n FROM {table}", conn).iloc[0]["n"]
    print(f"{table}: {count:,} rows")

# Check position values
print("\nPositions:", pd.read_sql("SELECT DISTINCT position FROM player_stats", conn)["position"].tolist())

# Check team_id values
print("Teams:", pd.read_sql("SELECT DISTINCT team_id FROM player_stats", conn)["team_id"].tolist())

# Check win values
print("Win values:", pd.read_sql("SELECT DISTINCT win FROM player_stats", conn)["win"].tolist())

# Null check on key columns
nulls = pd.read_sql("""
    SELECT
        SUM(CASE WHEN kills IS NULL THEN 1 ELSE 0 END) as kills_null,
        SUM(CASE WHEN position IS NULL THEN 1 ELSE 0 END) as position_null,
        SUM(CASE WHEN champion_id IS NULL THEN 1 ELSE 0 END) as champion_null
    FROM player_stats
""", conn)
print("\nNulls in player_stats:")
print(nulls)

conn.close()

matches: 497,102 rows
player_stats: 4,971,020 rows
player_items: 32,021,241 rows
player_runes: 29,826,120 rows
bans: 4,971,020 rows

Positions: ['TOP', 'JUNGLE', 'MIDDLE', 'BOTTOM', 'UTILITY', 'Invalid']
Teams: ['100', '200']
Win values: [0, 1]

Nulls in player_stats:
   kills_null  position_null  champion_null
0           0              0              0


In [2]:
"""
Run this from inside the analysis_service folder:
  python diagnose.py
"""
import sqlite3, os

DB = os.environ.get("LEAGUE_DB", "../dataset/league_data.db")
print(f"DB path: {DB}")
print(f"DB exists: {os.path.exists(DB)}")

conn = sqlite3.connect(DB)
c = conn.cursor()

print("\n--- team_stats columns ---")
c.execute("PRAGMA table_info(team_stats)")
ts_cols = [r[1] for r in c.fetchall()]
print(ts_cols)

print("\n--- player_stats columns ---")
c.execute("PRAGMA table_info(player_stats)")
ps_cols = [r[1] for r in c.fetchall()]
print(ps_cols)

print("\n--- team_stats team_id sample ---")
c.execute("SELECT DISTINCT team_id FROM team_stats LIMIT 5")
print(c.fetchall())

print("\n--- player_stats team_id sample ---")
c.execute("SELECT DISTINCT team_id FROM player_stats LIMIT 5")
print(c.fetchall())

print("\n--- player_stats position values ---")
c.execute("SELECT DISTINCT position FROM player_stats")
print(c.fetchall())

print("\n--- player_stats row count ---")
c.execute("SELECT COUNT(*) FROM player_stats")
print(c.fetchone())

print("\n--- team_stats row count ---")
c.execute("SELECT COUNT(*) FROM team_stats")
print(c.fetchone())

print("\n--- JOIN test (first 3 rows) ---")
try:
    c.execute("""
        SELECT ps.match_id, ps.champion_id, ps.team_id, ts.win
        FROM player_stats ps
        JOIN team_stats ts ON ps.match_id = ts.match_id AND ps.team_id = ts.team_id
        LIMIT 3
    """)
    rows = c.fetchall()
    print(rows)
    print(f"JOIN works, got {len(rows)} rows")
except Exception as e:
    print(f"JOIN failed: {e}")

print("\n--- win column in team_stats? ---")
print("win" in ts_cols)

conn.close()

DB path: ../dataset/league_data.db
DB exists: True

--- team_stats columns ---
['match_id', 'team_id', 'win', 'baron', 'dragon', 'tower', 'inhibitor', 'horde', 'first_blood', 'first_tower', 'first_dragon']

--- player_stats columns ---
['match_id', 'puuid', 'champion_id', 'team_id', 'win', 'position', 'kills', 'deaths', 'assists', 'gold', 'cs', 'dmg_champs', 'vision', 'kda', 'kp', 'summ1', 'summ2']

--- team_stats team_id sample ---
[('100',), ('200',)]

--- player_stats team_id sample ---
[('100',), ('200',)]

--- player_stats position values ---
[('TOP',), ('JUNGLE',), ('MIDDLE',), ('BOTTOM',), ('UTILITY',), ('Invalid',)]

--- player_stats row count ---
(4971020,)

--- team_stats row count ---
(994204,)

--- JOIN test (first 3 rows) ---
[('KR_7958577504', '14', '100', 0), ('KR_7958577504', '126', '100', 0), ('KR_7958577504', '103', '100', 0)]
JOIN works, got 3 rows

--- win column in team_stats? ---
True
